# Simple Binary Classification

Notebook ini membuat dataset sintetis, membagi data train-test, melatih model Logistic Regression, dan mengevaluasi hasil klasifikasi biner.

## 1. Persiapan dan import library

Jalankan sel instalasi berikut satu kali. Jika package sudah tersedia, pip akan melewati instalasi ulang.

In [ ]:
%pip install -q pandas matplotlib scikit-learn

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

## 2. Generate dataset sintetis

Dataset memiliki 1.000 observasi, 6 fitur numerik, dan target biner: `0` atau `1`.

In [ ]:
X_array, y_array = make_classification(
    n_samples=1_000,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    n_clusters_per_class=2,
    weights=[0.55, 0.45],
    class_sep=1.2,
    flip_y=0.03,
    random_state=RANDOM_STATE,
)

feature_names = [f"feature_{i}" for i in range(1, 7)]
df = pd.DataFrame(X_array, columns=feature_names)
df["target"] = y_array

print(f"Ukuran dataset: {df.shape}")
df.head()

## 3. Pemeriksaan data sederhana

In [ ]:
display(df.describe().T)

class_distribution = df["target"].value_counts().sort_index()
print("Jumlah missing value:", df.isna().sum().sum())
print("Distribusi kelas:")
display(class_distribution.rename("jumlah").to_frame())

class_distribution.plot(kind="bar", color=["steelblue", "darkorange"])
plt.title("Distribusi Target")
plt.xlabel("Kelas")
plt.ylabel("Jumlah Data")
plt.xticks(rotation=0)
plt.show()

## 4. Split data train dan test

Sebanyak 80% data digunakan untuk training dan 20% untuk testing. `stratify` menjaga proporsi kelas pada kedua bagian.

In [ ]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Data training: {X_train.shape}")
print(f"Data testing : {X_test.shape}")

## 5. Training model

Pipeline melakukan standardisasi fitur lalu melatih Logistic Regression. Standardisasi hanya dipelajari dari data training sehingga menghindari data leakage.

In [ ]:
model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)),
    ]
)

model.fit(X_train, y_train)
print("Model berhasil dilatih.")

## 6. Evaluasi model

In [ ]:
y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_probability)

print(f"Accuracy: {accuracy:.3f}")
print(f"ROC-AUC : {roc_auc:.3f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred),
    display_labels=model.classes_,
).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion Matrix")

RocCurveDisplay.from_predictions(y_test, y_probability, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--", label="Random classifier")
axes[1].set_title("ROC Curve")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Contoh hasil prediksi

Tabel berikut membandingkan label aktual, hasil prediksi, dan probabilitas kelas `1` untuk 10 data pertama pada test set.

In [ ]:
prediction_result = pd.DataFrame(
    {
        "actual": y_test.to_numpy(),
        "prediction": y_pred,
        "probability_class_1": y_probability,
    },
    index=y_test.index,
).sort_index()

prediction_result.head(10)

## Kesimpulan

Alur klasifikasi biner sederhana sudah mencakup generation data, pengecekan data, train-test split, preprocessing, training, dan evaluasi. Karena dataset bersifat sintetis, notebook ini cocok sebagai template awal sebelum menggantinya dengan data nyata.